<a href="https://colab.research.google.com/github/16pallavi16-oss/16pallavi16-oss/blob/main/Stock_analyser_ai_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Stock Advisor

Fetches live price/fundamentals via `yfinance`, recent news via NewsAPI, and asks a Groq-hosted LLM for a BUY/HOLD/AVOID recommendation.

**Run the cells top to bottom, in order.** You only need to run each one once.

## 1. Install packages

In [ ]:
!pip install -q groq yfinance requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.6 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import os
import re
from datetime import datetime, timedelta

import requests
import yfinance as yf
from groq import Groq

## 3. Load API keys from Colab Secrets

Click the 🔑 key icon in the left sidebar and add:
- `GROQ_API_KEY` — from https://console.groq.com/keys
- `NEWS_API_KEY` — from https://newsapi.org/register

Make sure **"Notebook access"** is toggled ON for both.

In [ ]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
    print("✅ API keys loaded from Colab secrets!")
except Exception as e:
    print("❌ Error loading secrets:", e)
    print("Add GROQ_API_KEY and NEWS_API_KEY via the 🔑 icon on the left, then re-run this cell.")
    raise

✅ API keys loaded from Colab secrets!


## 4. The StockAdvisor class

In [ ]:
def fmt_num(value):
    """Format a number with thousands separators; fall back to plain string
    for non-numeric values like 'N/A' (yfinance returns these often)."""
    if isinstance(value, (int, float)):
        return f"{value:,}"
    return str(value)


class StockAdvisor:
    def __init__(self, groq_key, news_key):
        self.groq = Groq(api_key=groq_key)
        self.news_key = news_key

    def get_stock_data(self, symbol):
        try:
            stock = yf.Ticker(symbol)
            info = stock.info

            if not info or info.get('regularMarketPrice') is None and info.get('currentPrice') is None:
                return None

            return {
                'symbol': symbol,
                'name': info.get('longName', symbol),
                'price': info.get('currentPrice', info.get('regularMarketPrice', 'N/A')),
                'change': info.get('regularMarketChangePercent', 'N/A'),
                'pe': info.get('trailingPE', 'N/A'),
                'volume': info.get('volume', 'N/A'),
                'market_cap': info.get('marketCap', 'N/A'),
                'sector': info.get('sector', 'N/A'),
                '52w_high': info.get('fiftyTwoWeekHigh', 'N/A'),
                '52w_low': info.get('fiftyTwoWeekLow', 'N/A'),
            }
        except Exception as e:
            print(f"Error fetching stock data: {e}")
            return None

    def get_news(self, company, symbol):
        try:
            url = "https://newsapi.org/v2/everything"
            params = {
                'q': f'{company} OR {symbol}',
                'from': (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d'),
                'sortBy': 'relevancy',
                'pageSize': 10,
                'language': 'en',
                'apiKey': self.news_key,
            }
            response = requests.get(url, params=params, timeout=15)
            response.raise_for_status()
            return response.json().get('articles', [])
        except Exception as e:
            print(f"Limited news data: {e}")
            return []

    def extract_symbol(self, query):
        extraction_prompt = f"""Extract the stock ticker symbol from this query: "{query}"

Common examples:
- "google" or "Google" -> GOOGL
- "apple" -> AAPL
- "microsoft" -> MSFT
- "tesla" -> TSLA
- "amazon" -> AMZN
- "meta" or "facebook" -> META
- "nvidia" -> NVDA

Respond with ONLY the stock ticker symbol in uppercase, nothing else."""

        response = self.groq.chat.completions.create(
          model="openai/gpt-oss-120b",
          messages=[
            {"role": "system", "content": "Respond with ONLY the ticker symbol. No words, no explanation, no punctuation."},
            {"role": "user", "content": extraction_prompt},
          ],
          temperature=0.1,
          max_tokens=200,
        )

        raw = response.choices[0].message.content.strip().upper()
        match = re.search(r'\b[A-Z]{1,5}\b', raw)
        symbol = match.group(0) if match else ""
        return symbol

    def analyze(self, query):
        try:
            symbol = self.extract_symbol(query)
            if not symbol or len(symbol) > 5:
                return "❌ Could not identify a valid stock symbol."
        except Exception as e:
            return f"❌ Error extracting symbol: {e}"

        print(f"📊 Fetching data for {symbol}...")
        stock = self.get_stock_data(symbol)
        if not stock:
            return f"❌ Could not fetch data for {symbol}. Check the ticker is correct."

        print(f"📰 Fetching news for {stock['name']}...")
        news = self.get_news(stock['name'], symbol)

        stock_info = f"""
**Company:** {stock['name']} ({stock['symbol']})
**Sector:** {stock['sector']}
**Current Price:** ${stock['price']}
**Change:** {stock['change']}%
**P/E Ratio:** {stock['pe']}
**Volume:** {fmt_num(stock['volume'])}
**Market Cap:** ${fmt_num(stock['market_cap'])}
**52-Week High:** ${stock['52w_high']}
**52-Week Low:** ${stock['52w_low']}
"""

        news_text = "\n".join(
            f"- {a.get('title', 'Untitled')} ({a.get('source', {}).get('name', 'Unknown source')})"
            for a in news[:8]
        ) or "No recent news found."

        analysis_prompt = f"""You are an expert financial analyst. Analyze this stock and provide investment guidance.

{stock_info}

Recent News (Last 7 Days):
{news_text}

Provide a clear analysis with:

1. **RECOMMENDATION:** BUY / HOLD / AVOID (be specific)
2. **KEY REASONS:** List 3-4 specific reasons supporting your recommendation
3. **NEWS SENTIMENT:** Positive/Negative/Neutral with brief explanation
4. **RISK LEVEL:** LOW / MEDIUM / HIGH
5. **PRICE OUTLOOK:** Short-term (1-3 months) price prediction

Be specific, data-driven, and helpful. Format clearly with headers."""

        try:
            print("🤖 AI analyzing...")
            response = self.groq.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": analysis_prompt}],
                temperature=0.3,
                max_tokens=1500,
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"❌ Analysis error: {e}"


## 5. Run it

Change the `query` variable to ask about any stock, then run this cell.

In [ ]:
advisor = StockAdvisor(GROQ_API_KEY, NEWS_API_KEY)

query = "Should I buy cisco stock?"  # <-- change this line to ask about a different stock

print("=" * 70)
print("🤖 AI STOCK ADVISOR")
print("=" * 70)
print(f"Query: {query}\n")

result = advisor.analyze(query)

print()
print("=" * 70)
print("📊 ANALYSIS RESULT:")
print("=" * 70)
print(result)


🤖 AI STOCK ADVISOR
Query: Should I buy cisco stock?

📊 Fetching data for CSCO...
📰 Fetching news for Cisco Systems, Inc....
🤖 AI analyzing...

📊 ANALYSIS RESULT:
**Cisco Systems, Inc. (CSCO) – Analyst Snapshot – 19 Jul 2026**

---

## 1. RECOMMENDATION  
**HOLD** (with a bias toward a modest **BUY** on pull‑backs)

*Rationale*: Cisco remains a cash‑generative, dividend‑paying heavyweight in networking and security. The stock is currently trading **mid‑range** of its 52‑week band and is slightly over‑valued on a trailing‑PE basis (≈ 37× vs. the 5‑year sector median of ~30×). The upside potential is primarily tied to the rollout of AI‑enabled networking and the expanding cybersecurity market, while downside risk is limited by the company’s strong balance sheet and resilient recurring‑revenue model.

---

## 2. KEY REASONS

| # | Reason | Supporting Data / Insight |
|---|--------|----------------------------|
| **1** | **Strong recurring‑revenue base** – 70%+ of FY‑2025 revenue came from 